# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/24f2001824/ml-flyrank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import pandas as pd
!git clone https://github.com/24f2001824/ml-flyrank.git
df = pd.read_csv("/content/ml-flyrank/data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

fatal: destination path 'ml-flyrank' already exists and is not an empty directory.
Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I will use Random Forest for this task.

The goal is to identify pages that are more likely to need a content refresh. Random Forest can handle several numeric and categorical features together and can capture non-linear relationships between the signals.

I will compare the model with the baseline using the same Precision@50 metric.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

features = [
    "search_volume",
    "competition",
    "competition_level",
    "cpc",
    "content_type",
    "main_intent",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

target = (df["trend_direction"] == "down").astype(int)

X = df[features].copy()
y = target.copy()

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent"
]

numeric_features = [
    col for col in features
    if col not in categorical_features
]

preprocessor = ColumnTransformer(
    [
        (
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_features
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        )
    ]
)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

print("Features:", len(features))
print("Target rate:", round(y.mean(), 3))

Features: 31
Target rate: 0.542


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will split the data by client instead of randomly splitting individual rows.

This keeps pages from the same client together and gives a more realistic test of how the model performs on clients it did not see during training.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

groups = df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

print("Train clients:", df.iloc[train_idx]["client_id"].nunique())
print("Test clients:", df.iloc[test_idx]["client_id"].nunique())

print(
    "Clients in both:",
    len(
        set(df.iloc[train_idx]["client_id"])
        & set(df.iloc[test_idx]["client_id"])
    )
)

Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Clients in both: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I will train the Random Forest on the training clients and evaluate it on the held-out clients.

The main metric is Precision@50 because the action is to review the highest-ranked pages first.

I will compare the model with the baseline using the same test pages.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
pipeline.fit(X_train, y_train)

model_scores = pipeline.predict_proba(X_test)[:, 1]

results = df.iloc[test_idx].copy()
results["model_score"] = model_scores
results["actual"] = y_test.values

def precision_at_k(scores, labels, k=50):
    order = np.argsort(scores)[::-1][:k]
    return labels.iloc[order].mean()

model_precision_50 = precision_at_k(
    results["model_score"].reset_index(drop=True),
    results["actual"].reset_index(drop=True),
    50
)

print("Model Precision@50:", round(model_precision_50, 3))

Model Precision@50: 1.0


In [7]:
results["baseline_score"] = (
    0.40 * results["impressions_90d"].rank(pct=True)
    + 0.30 * results["days_since_last_update"].rank(pct=True)
    + 0.25 * (
        1 - results["avg_position"].rank(pct=True)
    )
    + 0.05 * (
        1 - results["word_count"].rank(pct=True)
    )
)

baseline_precision_50 = precision_at_k(
    results["baseline_score"].reset_index(drop=True),
    results["actual"].reset_index(drop=True),
    50
)

comparison = pd.DataFrame({
    "Method": ["Baseline", "Random Forest"],
    "Precision@50": [
        baseline_precision_50,
        model_precision_50
    ]
})

display(comparison)

/usr/local/lib/python3.13/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: The behavior of Series.argsort in the presence of NA values is deprecated. In a future version, NA values will be ordered last instead of set to -1.
  return bound(*args, **kwds)


,Method,Precision@50
0,Baseline,0.36
1,Random Forest,1.00


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The model can still rank some pages incorrectly because the available features do not contain all the information that affects whether a page actually needs a refresh.

I will look at the top predictions and compare them with the observed label.

The feature importance gives an indication of which available signals the Random Forest relied on most. These should be treated as associations in the model, not causal effects.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top_predictions = results.sort_values(
    "model_score",
    ascending=False
).head(20)

display(
    top_predictions[
        [
            "content_id",
            "model_score",
            "actual",
            "impressions_90d",
            "clicks_90d",
            "ctr",
            "avg_position",
            "content_age_days",
            "days_since_last_update"
        ]
    ]
)

,content_id,model_score,actual,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,days_since_last_update
17707,content_f6bf66378677,0.985,1,817,0,0.00,5.7,141,20
23480,content_9234f5075e7a,0.980,1,1305,5,0.38,5.7,95,20
2139,content_41538bdb1b1e,0.980,1,205,0,0.00,7.3,97,8
10171,content_72f65ab80b1d,0.975,1,197,0,0.00,10.4,90,8
29681,content_9e8671965fff,0.970,1,1322,1,0.08,2.3,95,20
13454,content_0b50482209cd,0.970,1,10,0,0.00,5.7,96,8
3764,content_13f239ae855d,0.970,1,74,0,0.00,7.8,145,8
3652,content_c00a56f387ba,0.970,1,1,0,0.00,5.0,91,8
18564,content_bc3b1ade373b,0.970,1,194,2,1.03,7.6,141,8
18822,content_b23d650634c6,0.970,1,380,0,0.00,7.0,145,20


In [9]:
feature_names = pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

importance = pipeline.named_steps[
    "model"
].feature_importances_

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": importance
}).sort_values(
    "importance",
    ascending=False
)

display(feature_importance.head(15))

,feature,importance
18,numeric__impressions_prev_30d,0.191303
15,numeric__impressions_last_30d,0.156955
5,numeric__impressions_90d,0.077259
24,numeric__avg_position,0.059407
21,numeric__content_age_days,0.056388
13,numeric__days_with_impressions,0.050717
3,numeric__word_count,0.031225
4,numeric__char_count,0.031032
17,numeric__sessions_last_30d,0.027733
23,numeric__ctr,0.025809


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.